In [ ]:
# TGraph records carry attributes; convert to geometry only when needed.
def _as_topology(value):
    from topologicpy.Topology import Topology
    from topologicpy.Vertex import Vertex
    if isinstance(value, (list, tuple)):
        return [_as_topology(item) for item in value]
    if not isinstance(value, dict) or "index" not in value or "dictionary" not in value:
        return value
    representation = value.get("representation")
    attributes = value["dictionary"]
    if representation is None and "src" not in value:
        representation = Vertex.ByCoordinates(
            attributes.get("x", 0), attributes.get("y", 0), attributes.get("z", 0))
    if representation is None:
        raise ValueError("This graph record has no geometry representation")
    return Topology.SetDictionary(representation, attributes)


def _record_dictionary(value):
    from topologicpy.Topology import Topology
    if isinstance(value, dict) and "dictionary" in value:
        return value["dictionary"]
    return Topology.Dictionary(value)


def _edge_records(graph):
    from topologicpy.TGraph import TGraph
    records = TGraph.Edges(graph)
    if any(record.get("representation") is None for record in records):
        geometry = TGraph.Edges(graph, asTopologic=True, segmentCurves=False)
        if len(geometry) != len(records):
            raise ValueError("Could not generate one geometry representation per graph edge")
        for record, edge in zip(records, geometry):
            record["representation"] = edge
    return records


In [ ]:
# pip install openai chromadb numpy topologicpy sentence-transformers


In [ ]:
from topologicpy.TGraph import TGraph
from topologicpy.Vertex import Vertex
from topologicpy.Edge import Edge
from topologicpy.Dictionary import Dictionary

def graph_to_texts(graph):
    texts = []
    vertices = TGraph.Vertices(graph)
    for vertex in vertices:
        vertex_dict = Dictionary.PythonDictionary(_record_dictionary(vertex))
        vertex_label = vertex_dict.get('label', 'Unnamed Vertex')
        vertex_category = vertex_dict.get('category', 'Unknown Category')
        text = f"Vertex Label: {vertex_label}, Category: {vertex_category}, Coordinates: {Vertex.Coordinates(_as_topology(vertex))}"
        texts.append(text)
    edges = _edge_records(graph)
    for edge in edges:
        edge_dict = Dictionary.PythonDictionary(_record_dictionary(edge))
        edge_label = edge_dict.get('label', 'Unnamed Edge')
        start_vertex = Edge.StartVertex(_as_topology(edge))
        end_vertex = Edge.EndVertex(_as_topology(edge))
        text = f"Edge Label: {edge_label}, connects {Vertex.Coordinates(_as_topology(start_vertex))} and {Vertex.Coordinates(_as_topology(end_vertex))}"
        texts.append(text)
    return texts